In [ ]:
!pip install crewai crewai-tools langchain-google-genai

import os
import logging
import json
from crewai import Agent, Task, Crew
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

# Setup Gemini LLM
class ModelSetup:
    def __init__(self, model_name: str = "gemini-pro"):
        self.api_key = os.getenv("GEMINI_API_KEY", "your_gemini_api_key")
        if not self.api_key or self.api_key == "your_gemini_api_key":
            raise EnvironmentError("GEMINI_API_KEY not set")
        self.model_name = model_name

    def get_llm(self):
        return ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model=self.model_name,
            temperature=0.5,  # Changed from 0.7
            max_tokens=2048  # Added token limit
        )

# Sample code to analyze
sample_code = """
def factorial_recursive(n):
    if n < 0:
        return None
    if n == 0:
        return 1
    return n * factorial_recursive(n - 1)
"""

# Agents
syntax_checker = Agent(
    role="Syntax Validator",
    goal="Examine Python code for syntax errors and potential issues.",
    backstory="A meticulous Python expert focused on identifying syntax issues and edge cases.",
    llm=ModelSetup().get_llm(),
    verbose=True,
    memory=False
)

code_optimizer = Agent(
    role="Code Enhancer",
    goal="Optimize and correct code based on validation findings.",
    backstory="A seasoned developer skilled in improving Python code efficiency and reliability.",
    llm=ModelSetup().get_llm(),
    verbose=True,
    memory=False
)

# Tasks
validation_task = Task(
    description=f"""
    Review the following Python code for syntax errors, logical issues, or edge cases. Provide a JSON report listing any problems found:
    {sample_code}
    """,
    agent=syntax_checker,
    expected_output="A JSON string listing syntax or logical issues in the code."
)

optimization_task = Task(
    description=f"""
    Using the validation report, enhance the provided code to address identified issues and improve efficiency. Output only the optimized code as a string.
    Original code:
    {sample_code}
    """,
    agent=code_optimizer,
    expected_output="The optimized Python code as a string.",
    context=[validation_task]
)

# Create and run crew
def run_code_review():
    try:
        crew = Crew(
            agents=[syntax_checker, code_optimizer],
            tasks=[validation_task, optimization_task],
            verbose=True,
            process="sequential",
            memory=False
        )

        logging.debug("Initiating crew execution")
        result = crew.kickoff()
        logging.debug("Crew execution finished")

        # Format output as JSON
        output = {
            "validation": json.loads(validation_task.output.raw) if validation_task.output.raw else {},
            "optimized_code": optimization_task.output.raw
        }

        print("\n=== Code Review Report ===")
        print(json.dumps(output, indent=2))

        return output

    except Exception as e:
        logging.error(f"Execution failed: {str(e)}")
        error_output = {"error": f"Failed to process: {str(e)}"}
        print(json.dumps(error_output, indent=2))
        return error_output

if __name__ == "__main__":
    run_code_review()